# 02 Feature Selection: Correlation & VIF

1. Drop US-only columns and target leakage columns
2. Handle missing values
3. Remove high-correlation pairs (|r| > 0.85)
4. VIF screening (VIF > 10)

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import (
    load_data, set_seed, compute_vif, remove_high_corr_features,
    prepare_features, label_potential
)

set_seed(42)

## Load & Prepare

In [ ]:
df = load_data('../deepsolar_tract.csv')
target_col = 'tile_count'

# Drop US-only columns and target leakage columns, but KEEP tile_count
from src.features import get_target_cols
target_leakage = [c for c in get_target_cols() if c != target_col]

df_features = prepare_features(df, drop_us_only=True, drop_targets=False, drop_ids=True)
df_features = df_features.drop(columns=[c for c in target_leakage if c in df_features.columns])
print('Feature columns after dropping US-only/targets:', df_features.shape[1])

## Missing Value Imputation

In [ ]:
# Identify numeric columns
numeric_cols = df_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]  # exclude target to avoid duplicates
print(f'Numeric columns: {len(numeric_cols)}')

# Drop rows with missing target
df_clean = df_features[numeric_cols + [target_col]].dropna(subset=[target_col]).copy()

# Impute remaining missing values with median
for col in numeric_cols:
    if df_clean[col].isnull().any():
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f'Rows after dropping missing target: {len(df_clean)}')
print(f'Any remaining missing: {df_clean.isnull().sum().sum()}')

## Define Target (Tertile Classification)

In [ ]:
q33 = df_clean[target_col].quantile(0.33)
q66 = df_clean[target_col].quantile(0.66)
print(f'Quantiles: 33%={q33:.2f}, 66%={q66:.2f}')

df_clean['potential_label'] = label_potential(df_clean[target_col], q33, q66)
print(df_clean['potential_label'].value_counts().sort_index())

## Step 1: Correlation Screening

In [ ]:
feature_cols = [c for c in numeric_cols if c != target_col]
corr_selected = remove_high_corr_features(df_clean, feature_cols, target_col, threshold=0.85)
print(f'Features after correlation screening: {len(corr_selected)} (dropped {len(feature_cols) - len(corr_selected)})')

In [ ]:
# Visualize correlation heatmap for remaining features (top 30 by target corr)
target_corr = df_clean[corr_selected + [target_col]].corr()[target_col].abs().sort_values(ascending=False)
top30 = target_corr.head(31).index.tolist()  # includes target
top30.remove(target_col)

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(df_clean[top30].corr().abs(), cmap='coolwarm', center=0,
            square=True, ax=ax, xticklabels=True, yticklabels=True,
            vmin=0, vmax=1, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap: Top 30 Features (after high-corr removal)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig('../outputs/figures/02_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## Step 2: VIF Screening

In [ ]:
# Iterative VIF removal
remaining = corr_selected.copy()
vif_threshold = 10
max_iter = 50

for i in range(max_iter):
    vif_df = compute_vif(df_clean, remaining)
    max_vif = vif_df['VIF'].max()
    if max_vif <= vif_threshold:
        break
    drop_col = vif_df.iloc[0]['feature']
    remaining.remove(drop_col)
    print(f'Iter {i+1}: dropped {drop_col} (VIF={max_vif:.1f}), remaining={len(remaining)}')
else:
    print('Reached max iterations')

print(f'\nFinal feature count: {len(remaining)}')

In [ ]:
# Plot final VIF values
vif_final = compute_vif(df_clean, remaining)
fig, ax = plt.subplots(figsize=(10, max(6, len(vif_final)*0.3)))
colors = ['coral' if v > 10 else 'steelblue' for v in vif_final['VIF']]
ax.barh(range(len(vif_final)), vif_final['VIF'], color=colors)
ax.set_yticks(range(len(vif_final)))
ax.set_yticklabels(vif_final['feature'], fontsize=7)
ax.invert_yaxis()
ax.axvline(10, color='red', linestyle='--', label='VIF=10')
ax.set_xlabel('VIF')
ax.set_title(f'Final VIF Values ({len(vif_final)} features)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/02_vif_final.png', dpi=300, bbox_inches='tight')
plt.show()

## Save Selected Features

In [ ]:
selected_df = pd.DataFrame({'feature': remaining})
selected_df['target_corr'] = df_clean[remaining].corrwith(df_clean[target_col])
selected_df = selected_df.sort_values('target_corr', key=abs, ascending=False)
selected_df.to_csv('../outputs/results/selected_features.csv', index=False)
print('Saved to outputs/results/selected_features.csv')
print(selected_df.head(20))

## Export Clean Dataset for Modeling

In [ ]:
modeling_df = df_clean[remaining + ['potential_label']].copy()
modeling_df.to_csv('../data/processed/us_modeling_ready.csv', index=False)
print(f'Saved modeling dataset: {modeling_df.shape}')